In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q vllm

In [2]:
from datasets import load_dataset
from collections import Counter
ds = load_dataset("lmsys/mt_bench_human_judgments")
print(ds)
d = ds["human"]
print(d.column_names)
print({k: str(v)[:150] for k, v in d[0].items()})
print(Counter(d["winner"]))

DatasetDict({
    gpt4_pair: Dataset({
        features: ['question_id', 'model_a', 'model_b', 'winner', 'judge', 'conversation_a', 'conversation_b', 'turn'],
        num_rows: 2400
    })
    human: Dataset({
        features: ['question_id', 'model_a', 'model_b', 'winner', 'judge', 'conversation_a', 'conversation_b', 'turn'],
        num_rows: 3355
    })
})
['question_id', 'model_a', 'model_b', 'winner', 'judge', 'conversation_a', 'conversation_b', 'turn']
{'question_id': '81', 'model_a': 'alpaca-13b', 'model_b': 'gpt-3.5-turbo', 'winner': 'model_b', 'judge': 'author_2', 'conversation_a': "[{'content': 'Compose an engaging travel blog post about a recent trip to Hawaii, highlighting cultural experiences and must-see attractions.', 'role'", 'conversation_b': "[{'content': 'Compose an engaging travel blog post about a recent trip to Hawaii, highlighting cultural experiences and must-see attractions.', 'role'", 'turn': '1'}
Counter({'model_a': 1293, 'model_b': 1282, 'tie': 780})


In [3]:
import json, ast, numpy as np
from collections import Counter
from datasets import load_dataset

d = load_dataset("lmsys/mt_bench_human_judgments")["human"]

def msgs(c): return ast.literal_eval(c) if isinstance(c, str) else c
def fmt(c): return "\n\n".join(("[User]: " if m["role"]=="user" else "[Assistant]: ") + m["content"].strip() for m in msgs(c))
def last_len(c): return len([m for m in msgs(c) if m["role"]=="assistant"][-1]["content"])

pairs, votes = {}, []
for r in d:
    k = (str(r["question_id"]), r["model_a"], r["model_b"], str(r["turn"]))
    if k not in pairs:
        pairs[k] = {"a": fmt(r["conversation_a"]), "b": fmt(r["conversation_b"]),
                    "len_a": last_len(r["conversation_a"]), "len_b": last_len(r["conversation_b"])}
    votes.append((k, r["winner"]))

print("unique pairs:", len(pairs), "| human votes:", len(votes))
print("votes per pair:", Counter(Counter(k for k, _ in votes).values()))

SYS = "You are an impartial judge of AI assistant responses."
def judge_prompt(x, y):
    return ("Compare the two assistant conversations below. They answer the same user questions.\n\n"
            f"=== Assistant A ===\n{x}\n=== End of A ===\n\n=== Assistant B ===\n{y}\n=== End of B ===\n\n"
            "Which assistant answered the user better overall? Reply with a single letter: A or B.")

unique pairs: 2396 | human votes: 3355
votes per pair: Counter({1: 1635, 2: 599, 3: 132, 4: 24, 5: 6})


In [4]:
from vllm import LLM, SamplingParams
llm = LLM("Qwen/Qwen2.5-7B-Instruct-AWQ", quantization="awq", dtype="float16",
          gpu_memory_utilization=0.90, max_model_len=4096, enforce_eager=True)
tok = llm.get_tokenizer()
ids = [tok.encode(l, add_special_tokens=False)[0] for l in ("A", "B")]

def chat(u):
    return tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":u}],
                                   tokenize=False, add_generation_prompt=True)

prompts, meta = [], []
for k, p in pairs.items():
    for order in ("AB", "BA"):
        x, y = (p["a"], p["b"]) if order == "AB" else (p["b"], p["a"])
        prompts.append(chat(judge_prompt(x, y))); meta.append((k, order))

too_long = {meta[i][0] for i, s in enumerate(prompts) if len(tok.encode(s)) > 3900}
idx = [i for i, (k, _) in enumerate(meta) if k not in too_long]
print("pairs skipped (too long):", len(too_long))

outs = llm.generate([prompts[i] for i in idx], SamplingParams(temperature=0, max_tokens=1, logprobs=20))

res, missing = {}, 0
for i, o in zip(idx, outs):
    k, order = meta[i]; lp = o.outputs[0].logprobs[0]
    la = lp[ids[0]].logprob if ids[0] in lp else -1e9
    lb = lp[ids[1]].logprob if ids[1] in lp else -1e9
    missing += (ids[0] not in lp) or (ids[1] not in lp)
    p_first = 1 / (1 + np.exp(np.clip(lb - la, -50, 50)))     # P(first-listed answer wins)
    res.setdefault(k, {})[order] = float(p_first if order == "AB" else 1 - p_first)  # store P(model_a wins)
print("missing A/B logprob:", missing, "of", len(idx))

out = [{"key": list(k), "pA_ab": v["AB"], "pA_ba": v["BA"], "len_a": pairs[k]["len_a"], "len_b": pairs[k]["len_b"]}
       for k, v in res.items()]
json.dump({"pairs": out, "votes": [(list(k), w) for k, w in votes]}, open("/kaggle/working/judge_qwen7b.json", "w"))
print("saved", len(out), "pairs")

(MainProcess pid=319) INFO 10-05 18:33:15 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

(MainProcess pid=319) INFO 10-05 18:33:30 [model.py:699] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=319) INFO 10-05 18:33:30 [model.py:2109] Using max model len 4096
(MainProcess pid=319) INFO 10-05 18:33:31 [scheduler.py:308] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json:   0%|          | 0.00/62.7k [00:00<?, ?B/s]

Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=428) INFO 10-05 18:34:16 [default_loader.py:484] Loading weights took 1.65 seconds
(EngineCore pid=428) INFO 10-05 18:34:18 [model_runner.py:407] Model loading took 5.29 GiB memory and 39.807382 seconds
(EngineCore pid=428) WARNING 10-05 18:34:19 [topk_topp_sampler.py:131] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=428) INFO 10-05 18:34:19 [torch_utils.py:286] Reducing Torch threads from 2 to 1 for serving. Set OMP_NUM_THREADS in the external environment to override.
(EngineCore pid=428) INFO 10-05 18:34:19 [utils.py:343] Using LBNHC KV cache layout.
(EngineCore pid=428) INFO 10-05 18:34:29 [gpu_worker.py:692] Available KV cache memory: 6.44 GiB
(EngineCore pid=428) INFO 10-05 18:34:29 [kv_cache_utils.py:2464] GPU KV cache size: 120,576 tokens, Maximum concurrency for 4,096 tokens per request: 29.44x
(EngineCore pid=428) INFO 10-05 18:34:29 [gpu_worker.py:95

Rendering prompts:   0%|          | 0/4792 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 4792/4792 [22:37<00:00,  3.53it/s, est. speed input: 3810.19 toks/s, output: 3.53 toks/s]

missing A/B logprob: 0 of 4792
saved 2396 pairs


In [5]:
import json, numpy as np
from collections import defaultdict

J = json.load(open("/kaggle/working/judge_qwen7b.json"))
hv = defaultdict(lambda: [0, 0, 0])                      # human votes: model_a, model_b, tie
for k, w in J["votes"]:
    hv[tuple(k)][{"model_a": 0, "model_b": 1, "tie": 2}[w]] += 1
P = J["pairs"]; keys = [tuple(p["key"]) for p in P]
ab = np.array([p["pA_ab"] for p in P]); ba = np.array([p["pA_ba"] for p in P])   # P(model_a wins), a shown first / second
la = np.array([p["len_a"] for p in P]); lb = np.array([p["len_b"] for p in P])
h = np.array([hv[k][0] - hv[k][1] for k in keys])        # >0: humans prefer model_a (non-tie votes)
N = len(P); rng = np.random.default_rng(0); B = 2000
ci = lambda x: np.percentile(x, [2.5, 97.5]).round(3)

# ---- agreement with humans (pairs with a non-tied human majority) ----
L = np.where(h != 0)[0]; n = len(L); hA = h[L] > 0
a1, a2 = ab[L] > .5, ba[L] > .5; avg = (ab[L] + ba[L]) / 2 > .5; agree = a1 == a2
credit = {"single order (model_a first)": (a1 == hA).astype(float),
          "single order (model_a second)": (a2 == hA).astype(float),
          "both orders, tie if they disagree": np.where(agree, a1 == hA, 0.5),
          "both orders, averaged prob": (avg == hA).astype(float)}
decided = {k: np.ones(n) for k in credit}; decided["both orders, tie if they disagree"] = agree.astype(float)
idx = rng.integers(0, n, (B, n))
print(f"labeled pairs: {n} of {N} (ties/no majority dropped)\n")
print(f"{'method':36s} acc(tie=0.5)  95% CI          acc(decided)  coverage")
for k, c in credit.items():
    m = c[idx].mean(1); d = decided[k]
    print(f"{k:36s} {c.mean():.3f}       {ci(m)}  {(c*d).sum()/d.sum():.3f}        {d.mean():.3f}")
base = credit["single order (model_a first)"]
for k in ["both orders, averaged prob", "both orders, tie if they disagree"]:
    diff = (credit[k] - base)[idx].mean(1)
    print(f"paired diff vs single(model_a first): {k}: {(credit[k]-base).mean():+.3f} {ci(diff)}")

# ---- position bias (all pairs) ----
ia = rng.integers(0, N, (B, N))
flip = ((ab > .5) != (ba > .5)).astype(float)
first = np.concatenate([(ab > .5), (ba < .5)]).astype(float)       # verdict picks the answer shown first
first_pair = ((ab > .5).astype(float) + (ba < .5).astype(float)) / 2
print(f"\nflip rate (verdict changes with order): {flip.mean():.3f} {ci(flip[ia].mean(1))}")
print(f"picks first-shown answer: {first.mean():.3f} {ci(first_pair[ia].mean(1))}  (0.5 = no position bias)")

# ---- length bias (labeled pairs where one answer is >=1.5x longer) ----
ratio = np.maximum(la, lb) / np.maximum(np.minimum(la, lb), 1)
S = np.where((h != 0) & (ratio >= 1.5))[0]; ns = len(S); lng = la[S] > lb[S]
jud_long = (((ab[S] + ba[S]) / 2 > .5) == lng).astype(float); hum_long = ((h[S] > 0) == lng).astype(float)
i2 = rng.integers(0, ns, (B, ns))
print(f"\nlength-skewed pairs: {ns}")
print(f"judge picks longer answer: {jud_long.mean():.3f} {ci(jud_long[i2].mean(1))}")
print(f"humans pick longer answer: {hum_long.mean():.3f} {ci(hum_long[i2].mean(1))}")
print(f"judge - human: {(jud_long-hum_long).mean():+.3f} {ci((jud_long-hum_long)[i2].mean(1))}")
corr = (((ab[S] + ba[S]) / 2 > .5) == (h[S] > 0)).astype(float)
print(f"judge accuracy when humans prefer the longer answer: {corr[hum_long==1].mean():.3f} | the shorter: {corr[hum_long==0].mean():.3f}")

labeled pairs: 1858 of 2396 (ties/no majority dropped)

method                               acc(tie=0.5)  95% CI          acc(decided)  coverage
single order (model_a first)         0.791       [0.772 0.809]  0.791        1.000
single order (model_a second)        0.785       [0.766 0.803]  0.785        1.000
both orders, tie if they disagree    0.788       [0.772 0.804]  0.847        0.829
both orders, averaged prob           0.811       [0.793 0.828]  0.811        1.000
paired diff vs single(model_a first): both orders, averaged prob: +0.020 [0.008 0.032]
paired diff vs single(model_a first): both orders, tie if they disagree: -0.003 [-0.013  0.006]

flip rate (verdict changes with order): 0.193 [0.178 0.209]
picks first-shown answer: 0.479 [0.47  0.487]  (0.5 = no position bias)

length-skewed pairs: 1368
judge picks longer answer: 0.839 [0.818 0.859]
humans pick longer answer: 0.757 [0.733 0.779]
judge - human: +0.083 [0.064 0.102]
judge accuracy when humans prefer the longer answ

In [6]:
m = np.abs((ab[L] + ba[L]) / 2 - .5)               # margin of the averaged probability
correct = (avg == hA).astype(float); flipped = ~agree
for name, mask in [("order-consistent", agree), ("order-flipped", flipped)]:
    print(f"{name:17s} n={mask.sum():4d}  single-order acc {(a1 == hA)[mask].mean():.3f} | averaged acc {correct[mask].mean():.3f}")
srt = np.argsort(-m)
print("\nkeep the most confident fraction (by margin) -> accuracy")
for f in [1.0, .8, .6, .4, .2]:
    print(f"{f:.1f}  acc {correct[srt[:int(f * n)]].mean():.3f}")
from sklearn.metrics import roc_auc_score
print("AUROC of margin for predicting agreement with humans:", round(roc_auc_score(correct, m), 3))

order-consistent  n=1541  single-order acc 0.847 | averaged acc 0.847
order-flipped     n= 317  single-order acc 0.517 | averaged acc 0.634

keep the most confident fraction (by margin) -> accuracy
1.0  acc 0.811
0.8  acc 0.859
0.6  acc 0.905
0.4  acc 0.929
0.2  acc 0.957
AUROC of margin for predicting agreement with humans: 0.729


In [7]:
mc = m[agree]; cc = correct[agree]
print("margin AUROC within order-consistent pairs:", round(roc_auc_score(cc, mc), 3), "| n =", agree.sum())
idx2 = rng.integers(0, n, (B, n))
for f in [1.0, .6, .2]:
    k = int(f * n); accs = [correct[i[np.argsort(-m[i])[:k]]].mean() for i in idx2[:500]]
    print(f"keep {f:.1f}: acc {correct[srt[:k]].mean():.3f} 95% CI {np.percentile(accs, [2.5, 97.5]).round(3)}")

margin AUROC within order-consistent pairs: 0.72 | n = 1541
keep 1.0: acc 0.811 95% CI [0.793 0.829]
keep 0.6: acc 0.905 95% CI [0.889 0.921]
keep 0.2: acc 0.957 95% CI [0.935 0.977]
